# VeriDocs AI — VLM Supervised Fine-Tuning with LoRA (Qwen2-VL-2B-Instruct)

This notebook trains a parameter-efficient Low-Rank Adaptation (LoRA) adapter on **Qwen2-VL-2B-Instruct** on a **Free Google Colab T4 GPU**.

### Key Design & Constraint Choices:
- **Hardware**: Google Colab Free Tier T4 GPU (16 GB VRAM).
- **Quantization**: 4-bit NormalFloat (`bitsandbytes`) with `bnb_4bit_compute_dtype=torch.float16`.
- **Attention Mechanism**: `attn_implementation="sdpa"` (T4 does not support FlashAttention-2 or bf16).
- **Vision Tower**: Frozen to preserve generalized visual spatial awareness and save VRAM.
- **LoRA Parameters**: `r=16, lora_alpha=32, lora_dropout=0.05` applied to LM projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- **Tuning Discipline**: Trained strictly on **SEEN** layouts. Evaluation will test both seen and **UNSEEN** layouts.

In [ ]:
# Step 1: Install free open-source dependencies on Colab
!pip install -q --upgrade transformers peft bitsandbytes accelerate torchvision pypdfium2

In [ ]:
# Step 2: Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Load Model in 4-bit
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2-VL-2B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(model_id, max_pixels=1024 * 28 * 28)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

# Freeze vision tower
for param in model.visual.parameters():
    param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Step 4: Export Predictions
Once training completes, generate predictions for test documents and export them to `data/predictions/` in your local VeriDocs repo.